Historical workflow. Use `run_seeds_colab_pro.ipynb` for the corrected
implementation and verified input restoration. The cells below preserve an
older procedure and do not provide the current run identity guarantees.
Do not use them to resume or replace historical results.

# Multi-seed matched sweep (seeds 43–46)

Reruns `scripts/run_matched_sweep.py` with four more seeds; seed 42 is already in `experiments/matched/`.
The protocol and configs are unchanged; only the seed (fold assignment and initialisation) varies.

1. Runtime → Change runtime type → **T4 GPU**.
2. Put `bmpb-bundle.tar.gz` in Google Drive at `MyDrive/bmpb/` (it holds the dataset — keep it private).
3. Run the cells in order. Each finished run is saved to Drive, so after a disconnect just rerun all cells: completed runs are skipped.

Roughly 50 minutes per seed on a T4. Bring back `MyDrive/bmpb/seeds/` (or the downloaded archive) and unpack it into `experiments/seeds/`.

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before continuing"
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/bmpb"
import os
assert os.path.exists(f"{DRIVE}/bmpb-bundle.tar.gz"), "Upload bmpb-bundle.tar.gz to MyDrive/bmpb/ first"

In [ ]:
import json, subprocess, sys, tarfile
WORK = "/content/bmpb"
if not os.path.exists(f"{WORK}/bundle-manifest.json"):
    os.makedirs(WORK, exist_ok=True)
    with tarfile.open(f"{DRIVE}/bmpb-bundle.tar.gz") as archive:
        archive.extractall(WORK)
os.chdir(WORK)
print(json.load(open("bundle-manifest.json")))
pip = [sys.executable, "-m", "pip", "install", "-q"]
subprocess.run(pip + ["--no-deps", "-e", "."], check=True)
subprocess.run(pip + ["typer", "rich", "git+https://github.com/csebuetnlp/normalizer"], check=True)

In [ ]:
SEEDS = [43, 44, 45, 46]
env = {**os.environ, "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false"}
for seed in SEEDS:
    out = f"{DRIVE}/seeds/seed-{seed}"
    print(f"##### seed {seed} -> {out}", flush=True)
    subprocess.run([sys.executable, "-u", "scripts/run_matched_sweep.py", "--seed", str(seed), "--out", out],
                   check=True, env=env)
    subprocess.run(["cp", "bundle-manifest.json", out], check=True)

In [ ]:
for seed in SEEDS:
    rows = json.load(open(f"{DRIVE}/seeds/seed-{seed}/sweep_summary.json"))
    print(seed, len(rows), "runs,", sum(r["status"] == "ok" for r in rows), "ok")
with tarfile.open("/content/bmpb-seeds.tar.gz", "w:gz") as archive:
    archive.add(f"{DRIVE}/seeds", arcname="seeds")
from google.colab import files
files.download("/content/bmpb-seeds.tar.gz")